# 🐟 PhishBowl — Email Triage Console

A thin analyst console. You enter a target **mailbox**; the triage agent running on
**Amazon Bedrock AgentCore** investigates the mail that mailbox received recently
(headers, links, IPs, domains) and returns a confidence-scored report.

This notebook does **no local inference** — it just invokes the AgentCore runtime and
renders the result.

**How to use:** run the cells top-to-bottom. Edit the **target** in *Step 1*, then run
*Step 2* to trigger triage and *Step 3* to view the report.

## Setup — resolve the runtime coordinates

In [ ]:
import os, json, uuid, boto3

# The SageMaker lifecycle config writes these at instance start.
ENV_FILE = os.path.expanduser('~/SageMaker/phishbowl.env')
cfg = {}
if os.path.exists(ENV_FILE):
    for line in open(ENV_FILE):
        line = line.strip()
        if line and '=' in line and not line.startswith('#'):
            k, v = line.split('=', 1)
            cfg[k.strip()] = v.strip()

# Fall back to environment variables (handy for local runs).
RUNTIME_ARN = cfg.get('PHISHBOWL_RUNTIME_ARN') or os.environ.get('PHISHBOWL_RUNTIME_ARN', '')
REGION      = cfg.get('AWS_REGION') or os.environ.get('AWS_REGION', 'us-east-1')
ENDPOINT    = cfg.get('PHISHBOWL_ENDPOINT_NAME') or os.environ.get('PHISHBOWL_ENDPOINT_NAME', '')

assert RUNTIME_ARN, (
    'No PHISHBOWL_RUNTIME_ARN found. Check ~/SageMaker/phishbowl.env or set it manually below.'
)
# If you ever need to set it by hand, uncomment:
# RUNTIME_ARN = 'arn:aws:bedrock-agentcore:us-east-1:123456789012:runtime/phishbowl_agent-xxxx'

agentcore = boto3.client('bedrock-agentcore', region_name=REGION)
print('Region      :', REGION)
print('Runtime ARN :', RUNTIME_ARN)

## Step 1 — Enter the target

Set the mailbox to triage and (optionally) the look-back window. Defaults match the
'last 5 minutes' trigger; raise `LOOKBACK_MINUTES` or `MAX_MESSAGES` to widen the sweep.

> The agent queries Sublime by **mailbox (email address)**. If you only have a display
> name, resolve it to an email first (e.g. via your directory).

In [ ]:
TARGET          = 'jdoe@corp.example'   # <-- the user's email address to triage
LOOKBACK_MINUTES = 5                      # how far back to look
MAX_MESSAGES     = 25                     # cap for cost / latency

## Step 2 — Trigger the triage workflow

Invokes the AgentCore runtime. The agent pulls recent mail from Sublime, runs its
header / link / IP analysis tools, and returns a structured report.

In [ ]:
def read_agentcore_response(response):
    """Return the raw response text, handling json and event-stream content types."""
    ctype = response.get('contentType', '') or ''
    body = response['response']
    if 'text/event-stream' in ctype:
        parts = []
        for line in body.iter_lines(chunk_size=1024):
            if not line:
                continue
            s = line.decode('utf-8') if isinstance(line, (bytes, bytearray)) else line
            if s.startswith('data:'):
                s = s[len('data:'):].strip()
            parts.append(s)
        return ''.join(parts)
    raw = body.read()
    return raw.decode('utf-8') if isinstance(raw, (bytes, bytearray)) else raw


payload = json.dumps({
    'mailbox': TARGET,
    'minutes': LOOKBACK_MINUTES,
    'max_messages': MAX_MESSAGES,
}).encode('utf-8')

print(f'Triaging {TARGET} (last {LOOKBACK_MINUTES} min)... this can take a moment.')
resp = agentcore.invoke_agent_runtime(
    agentRuntimeArn=RUNTIME_ARN,
    runtimeSessionId=uuid.uuid4().hex + uuid.uuid4().hex,  # >=33 chars
    payload=payload,
)

raw = read_agentcore_response(resp)
outer = json.loads(raw)
report = outer.get('report', {})
narrative = outer.get('narrative', '')
if report.get('_parse_error'):
    print('WARNING: agent did not return clean JSON:', report['_parse_error'])
print('Messages analyzed:', report.get('messages_analyzed', 'n/a'))

## Step 3 — The triage report

In [ ]:
from IPython.display import HTML, display

VERDICT_COLOR = {
    'malicious': '#b91c1c',
    'likely_phishing': '#dc2626',
    'suspicious': '#d97706',
    'benign': '#16a34a',
}
CONF_BG = {'HIGH': '#111827', 'MEDIUM': '#374151', 'LOW': '#6b7280', 'INFO': '#9ca3af'}

def _badge(text, bg):
    return (f"<span style='background:{bg};color:#fff;padding:2px 8px;border-radius:10px;"
            f"font-size:12px;font-weight:600'>{text}</span>")

def render(report):
    if not report or report.get('_parse_error'):
        return HTML('<pre>' + (report.get('raw', 'No report.') if report else 'No report.') + '</pre>')
    o = report.get('overall', {})
    vc = VERDICT_COLOR.get(o.get('verdict', ''), '#374151')
    html = ["<div style=\"font-family:-apple-system,Segoe UI,Roboto,sans-serif;max-width:900px\">"]
    html.append(f"<h2 style='margin-bottom:4px'>🐟 Triage: {report.get('mailbox','')}</h2>")
    html.append(f"<div style='color:#6b7280;margin-bottom:10px'>Window: last {report.get('window_minutes','?')} min &middot; "
                f"{report.get('messages_analyzed',0)} message(s) analyzed</div>")
    html.append("<div style='padding:12px;border-left:5px solid " + vc + ";background:#f9fafb;margin-bottom:16px'>")
    html.append('Overall: ' + _badge(o.get('verdict','?').upper(), vc) + ' ' +
                _badge('confidence ' + o.get('confidence','?'), CONF_BG.get(o.get('confidence',''), '#374151')))
    html.append(f"<div style='margin-top:8px'>{o.get('summary','')}</div></div>")
    for f in report.get('findings', []):
        fvc = VERDICT_COLOR.get(f.get('verdict',''), '#374151')
        html.append("<div style='border:1px solid #e5e7eb;border-radius:8px;padding:12px;margin-bottom:10px'>")
        html.append(f"<div style='font-weight:600'>{f.get('subject','(no subject)')}</div>")
        html.append(f"<div style='color:#6b7280;font-size:13px;margin-bottom:6px'>from {f.get('sender','?')}</div>")
        html.append(_badge(f.get('verdict','?').upper(), fvc) + ' ' +
                    _badge('confidence ' + f.get('confidence','?'), CONF_BG.get(f.get('confidence',''), '#374151')))
        sigs = ''.join(f"<li>{s}</li>" for s in f.get('key_signals', []))
        if sigs:
            html.append(f"<ul style='margin:8px 0'>{sigs}</ul>")
        html.append(f"<div style='font-size:13px'><b>Why:</b> {f.get('rationale','')}</div>")
        html.append(f"<div style='font-size:13px;margin-top:4px'><b>Action:</b> {f.get('recommended_action','')}</div>")
        html.append('</div>')
    html.append('</div>')
    return HTML(''.join(html))

display(render(report))

### Model reasoning & raw JSON

The agent's narrative trajectory and the raw structured report, for the record.

In [ ]:
print(narrative)
print('\n' + '=' * 60 + '\nRAW REPORT JSON\n' + '=' * 60)
print(json.dumps(report, indent=2))